# Benchmark functions

In [17]:
import random
import math

random.seed(23)

def sphere(x):
    return sum(xi**2 for xi in x)

def schwefel1(x):
    total = 0
    for i in range(len(x)):
        for j in range(i + 1):
            total += x[j]**2
    return total

def rastrigin(x):
    n = len(x)
    return 10 * n + sum(xi**2 - 10 * math.cos(2 * math.pi * xi) for xi in x)

def rosenbrock(x):
    total = 0
    for i in range(len(x) - 1):
        total += 100 * (x[i+1] - x[i]**2)**2 + (1 - x[i])**2
    return total

def schwefel(x):
    return sum(-xi * math.sin(math.sqrt(abs(xi))) for xi in x)

# PSO algorithm for comparison with SFA

In [ ]:
def pso(fitness_func, lower, upper, n_dims=10,
        swarm_size=30, iterations=500,
        w=0.7, c1=1.6, c2=1.6):

    
    particles = [[random.uniform(lower, upper) for _ in range(n_dims)]
                 for _ in range(swarm_size)]
    velocities = [[random.uniform(lower, upper) for _ in range(n_dims)]
                  for _ in range(swarm_size)]
    fitnesses = [fitness_func(p) for p in particles]

    personal_best = [p.copy() for p in particles]
    personal_best_fitness = fitnesses.copy()

    best_idx = min(range(swarm_size), key=lambda i: fitnesses[i])
    global_best = particles[best_idx].copy()
    global_best_fitness = fitnesses[best_idx]

    for _ in range(iterations):
        r1 = random.random()
        r2 = random.random()

        for i in range(swarm_size):
            velocities[i] = [
                w * velocities[i][d]
                + c1 * r1 * (personal_best[i][d] - particles[i][d])
                + c2 * r2 * (global_best[d] - particles[i][d])
                for d in range(n_dims)
            ]
            particles[i] = [
                max(lower, min(upper, particles[i][d] + velocities[i][d]))
                for d in range(n_dims)
            ]

            fit = fitness_func(particles[i])
            if fit < personal_best_fitness[i]:
                personal_best[i] = particles[i].copy()
                personal_best_fitness[i] = fit

            if fit < global_best_fitness:
                global_best = particles[i].copy()
                global_best_fitness = fit

    return global_best_fitness

# SFA 

In [ ]:
def special_forces_algorithm(fitness_func, lower, upper,
                              n_dims=3,
                              population_size=30,
                              iterations=500,
                              tv1=0.5, tv2=0.3,
                              p0=0.25, k=0.1):

    soldiers = [
        [random.uniform(lower, upper) for _ in range(n_dims)]
        for _ in range(population_size)
    ]
    fitnesses = [fitness_func(s) for s in soldiers]

    best_idx = min(range(population_size), key=lambda i: fitnesses[i])
    global_best = soldiers[best_idx].copy()
    global_best_fitness = fitnesses[best_idx]

    for t in range(1, iterations + 1) :
        # Unmaned search
        
        c = k * (lower + (1 - t / iterations) * (upper - lower))
        uav_results = []
        for i in range(population_size):
            v = [random.gauss(0, 1) for _ in range(n_dims)]
            norm = math.sqrt(sum(vi**2 for vi in v))
            if norm == 0:
                norm = 1e-10
            v = [vi / norm * abs(c) for vi in v]
            xu = [soldiers[i][d] + v[d] for d in range(n_dims)]
            xu = [max(lower, min(upper, xu[d])) for d in range(n_dims)]
            uav_results.append(xu)

        for xu in uav_results:
            f_xu = fitness_func(xu)
            if f_xu < global_best_fitness:
                global_best = xu.copy()
                global_best_fitness = f_xu

        instruction = (1 - 0.15 * random.random()) * (1 - t / iterations)

        p_loss = p0 * math.cos(math.pi * t / (2 * iterations))

        w = 0.75 - 0.55 * math.atan((t / iterations) ** (2 * math.pi))

        x_ave = [
            sum(soldiers[i][d] for i in range(population_size)) / population_size
            for d in range(n_dims)
        ]

        new_soldiers = []

        for i in range(population_size):
            r1 = random.random()
            r2 = random.random()

            if instruction >= tv1:
                # PHASE 1: EXPLORATION

                if r1 >= 0.5:
                    # Large scale search
                    
                    sign = 1 if random.random() >= 0.5 else -1
                    new_pos = [
                        r1 * (global_best[d] - soldiers[i][d])
                        + sign * (1 - r1) * (upper - lower)
                        for d in range(n_dims)
                    ]
                else:
                    # Raid
                    
                    if random.random() < p_loss:
                        aim_idx = random.randint(0, population_size - 1)
                        x_aim = soldiers[aim_idx]
                        f_aim = fitnesses[aim_idx]
                    else:
                        x_aim = global_best
                        f_aim = global_best_fitness

                    f_i = fitnesses[i]
                    denom = f_i + f_aim
                    if denom == 0:
                        denom = 1e-10
                    ratio = f_i / denom

                    a_i = [ratio * (x_aim[d] - soldiers[i][d]) for d in range(n_dims)]
                    new_pos = [soldiers[i][d] + w * a_i[d] for d in range(n_dims)]

            elif tv2 < instruction < tv1:
                # PHASE 2: TRANSITION 

                if r2 >= 0.5:
                    # Raid 
                    if random.random() < p_loss:
                        aim_idx = random.randint(0, population_size - 1)
                        x_aim = soldiers[aim_idx]
                        f_aim = fitnesses[aim_idx]
                    else:
                        x_aim = global_best
                        f_aim = global_best_fitness

                    f_i = fitnesses[i]
                    denom = f_i + f_aim
                    if denom == 0:
                        denom = 1e-10
                    ratio = f_i / denom

                    a_i = [ratio * (x_aim[d] - soldiers[i][d]) for d in range(n_dims)]
                    new_pos = [soldiers[i][d] + w * a_i[d] for d in range(n_dims)]
                else:
                    # move toward global best 
                    new_pos = [
                        instruction * (global_best[d] - soldiers[i][d]) + 0.1 * soldiers[i][d]
                        for d in range(n_dims)
                    ]

            else:
                # PHASE 3: EXPLOITATION
                # converge around global best using average position as reference
                r = [random.uniform(-1, 1) for _ in range(n_dims)]
                new_pos = [
                    global_best[d] + r[d] * abs(global_best[d] - x_ave[d])
                    for d in range(n_dims)
                ]

            new_pos = [max(lower, min(upper, new_pos[d])) for d in range(n_dims)]
            new_soldiers.append(new_pos)

        soldiers = new_soldiers
        fitnesses = [fitness_func(s) for s in soldiers]

        for i in range(population_size):
            if fitnesses[i] < global_best_fitness:
                global_best = soldiers[i].copy()
                global_best_fitness = fitnesses[i]

        # if t % 10 == 0:
        #     if instruction >= tv1:
        #         phase = "1-Exploration"
        #     elif instruction > tv2:
        #         phase = "2-Transition"
        #     else:
        #         phase = "3-Exploitation"
        #     print(f"Iter {t}: best={global_best_fitness:.4e}, instruction={instruction:.3f}, phase={phase}")

    return global_best_fitness

In [ ]:
print(special_forces_algorithm(sphere,-5.12, 5.12, 50))

2.580721257698465e-113


# Comparison run between the 2 algorithms

In [40]:
def run_comparison():
    N_RUNS = 5

    configs = [
        ("Sphere",     sphere,     -5.12,   5.12,   50),
        ("Schwefel 1", schwefel1,  -65.536, 65.536, 50),
        ("Rastrigin",  rastrigin,  -5.12,   5.12,   50),
        ("Rosenbrock", rosenbrock, -65.536, 65.536, 50),
        ("Schwefel",   schwefel,   -65.536, 65.536, 50),
    ]

    print("=" * 65)
    print(f"{'Function':<14} {'SFA avg':>15} {'PSO avg':>15} {'Winner':>8}")
    print("=" * 65)

    for name, func, lower, upper, n_dims in configs:
        sfa_results = []
        pso_results = []

        for run in range(N_RUNS):
            sfa_fit = special_forces_algorithm(func, lower, upper,
                                                  n_dims=n_dims,
                                                  population_size=30,
                                                  iterations=500)
            sfa_results.append(sfa_fit)

            pso_fit = pso(func, lower, upper,
                          n_dims=n_dims,
                          swarm_size=30,
                          iterations=500)
            pso_results.append(pso_fit)

        sfa_avg = sum(sfa_results) / N_RUNS
        pso_avg = sum(pso_results) / N_RUNS
        winner = "SFA" if sfa_avg < pso_avg else "PSO"

        print(f"{name:<14} {sfa_avg:>15.4e} {pso_avg:>15.4e} {winner:>8}")

    print("=" * 65)


run_comparison()

Function               SFA avg         PSO avg   Winner
Sphere             5.0216e-101      6.2746e+01      SFA
Schwefel 1          5.2751e-96      1.4854e+05      SFA
Rastrigin           0.0000e+00      3.5865e+02      SFA
Rosenbrock          2.0314e-01      3.7416e+08      SFA
Schwefel           -3.1817e+03     -1.7781e+03      SFA
